# Notebook 04 — Embeddings y Transformers

## Objetivo

Comparar modelos más complejos sobre el **subconjunto político**:
- **Parte A:** Embeddings preentrenados (GloVe) + LR/SVM
- **Parte B:** Fine-tuning de DistilBERT

> Los modelos aprenden patrones lingüísticos del dataset; no verifican hechos.

In [10]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import joblib

from nlp.metrics import compute_metrics, metrics_to_row
from nlp.plotting import plot_confusion_matrix
from sklearn.metrics import confusion_matrix


## Parte A — Embeddings (GloVe)

In [12]:

pol_train = pd.read_csv(DATA_PROCESSED / 'politics_train.csv')
pol_val = pd.read_csv(DATA_PROCESSED / 'politics_val.csv')
pol_test = pd.read_csv(DATA_PROCESSED / 'politics_test.csv')

GLOVE_DIM = 100
GLOVE_URL = 'http://nlp.stanford.edu/data/glove.6B.zip'
GLOVE_DIR = DATA_EMBEDDINGS
GLOVE_DIR.mkdir(parents=True, exist_ok=True)


In [11]:

import urllib.request
import zipfile

def load_glove_vectors(dim=100):
    glove_file = GLOVE_DIR / f'glove.6B.{dim}d.txt'
    if not glove_file.exists():
        zip_path = GLOVE_DIR / 'glove.6B.zip'
        if not zip_path.exists():
            print('Descargando GloVe (puede tardar varios minutos)...')
            urllib.request.urlretrieve(GLOVE_URL, zip_path)
        print('Extrayendo GloVe...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extract(f'glove.6B.{dim}d.txt', GLOVE_DIR)

    vectors = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc='Cargando GloVe'):
            parts = line.strip().split()
            word = parts[0]
            vec = np.array(parts[1:], dtype=np.float32)
            vectors[word] = vec
    return vectors

glove = load_glove_vectors(GLOVE_DIM)
print(f'Vectores cargados: {len(glove):,}')


Cargando GloVe: 0it [00:00, ?it/s]

Vectores cargados: 400,000


In [13]:

def text_to_embedding(text, vectors, dim=100):
    tokens = str(text).lower().split()
    vecs = [vectors[t] for t in tokens if t in vectors]
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0)

def embed_corpus(texts, vectors, dim=100):
    return np.vstack([text_to_embedding(t, vectors, dim) for t in tqdm(texts, desc='Embedding')])

TEXT_COL = 'clean_full_text_without_stopwords'
X_train_emb = embed_corpus(pol_train[TEXT_COL].fillna(''), glove, GLOVE_DIM)
X_val_emb = embed_corpus(pol_val[TEXT_COL].fillna(''), glove, GLOVE_DIM)
X_test_emb = embed_corpus(pol_test[TEXT_COL].fillna(''), glove, GLOVE_DIM)
y_train, y_val, y_test = pol_train['label'], pol_val['label'], pol_test['label']


Embedding:   0%|          | 0/12635 [00:00<?, ?it/s]

Embedding:   0%|          | 0/2707 [00:00<?, ?it/s]

Embedding:   0%|          | 0/2708 [00:00<?, ?it/s]

In [14]:

embedding_results = []
for model_name, ModelClass, params in [
    ('logistic_regression', LogisticRegression, {'C': [0.1, 1, 10]}),
    ('linear_svm', LinearSVC, {'C': [0.1, 1, 10]}),
]:
    best_f2, best_clf, best_param = -1, None, None
    for C in params['C']:
        clf = ModelClass(C=C, random_state=RANDOM_STATE, max_iter=2000) if model_name == 'logistic_regression' else ModelClass(C=C, random_state=RANDOM_STATE)
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
        pipe.fit(X_train_emb, y_train)
        y_val_pred = pipe.predict(X_val_emb)
        m = compute_metrics(y_val, y_val_pred)
        if m['f2_fake'] > best_f2:
            best_f2, best_clf, best_param = m['f2_fake'], pipe, C

    y_test_pred = best_clf.predict(X_test_emb)
    y_proba = None
    if hasattr(best_clf.named_steps['clf'], 'predict_proba'):
        y_proba = best_clf.predict_proba(X_test_emb)[:, 1]
    elif hasattr(best_clf.named_steps['clf'], 'decision_function'):
        s = best_clf.decision_function(X_test_emb)
        y_proba = (s - s.min()) / (s.max() - s.min() + 1e-9)

    test_m = compute_metrics(y_test, y_test_pred, y_proba)
    embedding_results.append(metrics_to_row(test_m, {
        'model': model_name, 'representation': 'glove_avg',
        'best_param': best_param, 'dataset_scope': 'politics', 'split': 'test',
    }))

embedding_df = pd.DataFrame(embedding_results)
embedding_df.to_csv(RESULTS_METRICS / 'embedding_results.csv', index=False)
embedding_df


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,representation,best_param,dataset_scope,split
0,0.934638,0.898790,0.905765,0.902264,0.904361,0.974721,logistic_regression,glove_avg,1.0,politics,test
1,0.933900,0.899448,0.902439,0.900941,0.901839,0.974231,linear_svm,glove_avg,0.1,politics,test


## Parte B — Fine-tuning DistilBERT

> **Alcance:** solo subconjunto **politics** (no se entrena sobre el dataset completo).

In [15]:

# Parámetros (config única; ajustar si hay limitaciones computacionales)
SAMPLE_FRAC = 1.0
LEARNING_RATES = [2e-5]
BATCH_SIZES = [8]
EPOCHS_LIST = [2]
MAX_LENGTHS = [256]

# Reutilizar checkpoint ya entrenado (evita re-entrenar ~13h en CPU)
REUSE_EXISTING_CHECKPOINT = True
CHECKPOINT_DIR = RESULTS_MODELS / 'distilbert_checkpoints'

print(f'SAMPLE_FRAC={SAMPLE_FRAC}')
print('Alcance DistilBERT: politics solamente')


SAMPLE_FRAC=1.0
Alcance DistilBERT: politics solamente


In [16]:

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from sklearn.metrics import fbeta_score, accuracy_score, precision_recall_fscore_support

def prepare_transformer_input(row, max_chars=3000):
    title = str(row.get('title', ''))
    body = str(row.get('text', ''))
    full = f"{title} {body}".strip()
    if len(full) > max_chars:
        paragraphs = body.split('\n\n')
        short_body = paragraphs[0] if paragraphs else body[:2000]
        full = f"{title} {short_body}".strip()
    return full[:max_chars]

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_length)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    f2 = fbeta_score(labels, preds, beta=2, pos_label=1, zero_division=0)
    return {'f2_fake': f2, 'f1_fake': f1[1] if len(f1) > 1 else 0, 'accuracy': accuracy_score(labels, preds)}


In [17]:

# Preparar datos
tr = pol_train.copy()
va = pol_val.copy()
te = pol_test.copy()

if SAMPLE_FRAC < 1.0:
    tr = tr.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    va = va.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    print(f'Muestra reducida: train={len(tr)}, val={len(va)}')

tr['transformer_text'] = tr.apply(prepare_transformer_input, axis=1)
va['transformer_text'] = va.apply(prepare_transformer_input, axis=1)
te['transformer_text'] = te.apply(prepare_transformer_input, axis=1)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')


In [18]:

def find_best_checkpoint(checkpoint_dir):
    """Busca el mejor checkpoint guardado por Trainer (load_best_model_at_end)."""
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None
    best_path, best_metric = None, -1.0
    for state_file in checkpoint_dir.glob('checkpoint-*/trainer_state.json'):
        import json
        state = json.loads(state_file.read_text(encoding='utf-8'))
        metric = state.get('best_metric')
        path = state.get('best_model_checkpoint')
        if metric is not None and metric >= best_metric and path:
            best_metric = metric
            best_path = Path(path)
    return best_path

transformer_results = []
best_trainer, best_row, best_model = None, None, None
best_f2 = -1

for lr in LEARNING_RATES:
    for bs in BATCH_SIZES:
        for epochs in EPOCHS_LIST:
            for max_len in MAX_LENGTHS:
                print(f'\n=== lr={lr}, bs={bs}, epochs={epochs}, max_len={max_len} ===')
                train_ds = NewsDataset(tr['transformer_text'], tr['label'], tokenizer, max_len)
                val_ds = NewsDataset(va['transformer_text'], va['label'], tokenizer, max_len)
                test_ds = NewsDataset(te['transformer_text'], te['label'], tokenizer, max_len)

                checkpoint_path = find_best_checkpoint(CHECKPOINT_DIR)
                can_reuse = (
                    REUSE_EXISTING_CHECKPOINT
                    and checkpoint_path is not None
                    and checkpoint_path.exists()
                    and lr == 2e-5 and bs == 8 and epochs == 2 and max_len == 256
                )

                if can_reuse:
                    print(f'Reutilizando checkpoint existente: {checkpoint_path}')
                    model = AutoModelForSequenceClassification.from_pretrained(str(checkpoint_path))
                    trainer = None
                else:
                    model = AutoModelForSequenceClassification.from_pretrained(
                        'distilbert-base-uncased', num_labels=2
                    )
                    args = TrainingArguments(
                        output_dir=str(CHECKPOINT_DIR),
                        learning_rate=lr,
                        per_device_train_batch_size=bs,
                        per_device_eval_batch_size=bs,
                        num_train_epochs=epochs,
                        eval_strategy='epoch',
                        save_strategy='epoch',
                        load_best_model_at_end=True,
                        metric_for_best_model='f2_fake',
                        greater_is_better=True,
                        logging_steps=50,
                        seed=RANDOM_STATE,
                        report_to='none',
                    )
                    trainer = Trainer(
                        model=model, args=args,
                        train_dataset=train_ds, eval_dataset=val_ds,
                        compute_metrics=compute_transformer_metrics,
                        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
                    )
                    trainer.train()
                    model = trainer.model

                pred = Trainer(model=model).predict(test_ds)
                y_pred = np.argmax(pred.predictions, axis=-1)
                y_proba = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
                test_m = compute_metrics(te['label'], y_pred, y_proba)
                row = metrics_to_row(test_m, {
                    'model': 'distilbert-base-uncased',
                    'learning_rate': lr, 'batch_size': bs,
                    'epochs': epochs, 'max_length': max_len,
                    'sample_frac': SAMPLE_FRAC,
                    'dataset_scope': 'politics', 'split': 'test',
                    'reused_checkpoint': can_reuse,
                })
                transformer_results.append(row)
                print('Métricas test:', {k: round(v, 4) for k, v in test_m.items() if k != 'roc_auc'})
                if test_m['f2_fake'] > best_f2:
                    best_f2 = test_m['f2_fake']
                    best_trainer = trainer
                    best_row = row
                best_model = model

transformer_df = pd.DataFrame(transformer_results)
transformer_df.to_csv(RESULTS_METRICS / 'transformer_results.csv', index=False)
if best_model is not None:
    best_model.save_pretrained(str(RESULTS_MODELS / 'best_distilbert'))
    tokenizer.save_pretrained(str(RESULTS_MODELS / 'best_distilbert'))
    print('Modelo guardado en results/models/best_distilbert')
transformer_df.sort_values('f2_fake', ascending=False)



=== lr=2e-05, bs=8, epochs=2, max_len=256 ===
Reutilizando checkpoint existente: C:\Users\candi\Documentos\ITBA\QUINTO\NLP\nlp\results\models\distilbert_checkpoints\checkpoint-1580


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Métricas test: {'accuracy': 0.9989, 'precision_fake': 1.0, 'recall_fake': 0.9967, 'f1_fake': 0.9983, 'f2_fake': 0.9973}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en results/models/best_distilbert


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,learning_rate,batch_size,epochs,max_length,sample_frac,dataset_scope,split,reused_checkpoint
0,0.998892,1.0,0.996674,0.998334,0.997337,1.0,distilbert-base-uncased,0.00002,8,2,256,1.0,politics,test,True


## Conclusiones

- Los embeddings GloVe capturan semántica pero pueden perder señales de estilo.
- DistilBERT puede capturar contexto más rico; usar `SAMPLE_FRAC < 1.0` si hay limitaciones de hardware.
- Ningún modelo verifica hechos; aprenden patrones del dataset.